In [ ]:
# Install the PyPharao code and py3Dmol
!pip install rdkit
!pip install git+https://github.com/silicos-it/PyPharao.git
!pip install mols2grid py3Dmol

In [ ]:
# Import all required modules
from tqdm import tqdm_notebook as tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole

from IPython.display import display
from IPython.display import SVG

import py3Dmol
from ipywidgets import widgets

import mols2grid
from pypharao import *

from pathlib import Path
import requests

In [ ]:
# Define the URL of the file on GitHub
SMI_FILE_URL = "https://raw.githubusercontent.com/silicos-it/PyPharao/master/examples/datasets/compounds_10k.smi"

# Define the local path where the file should be saved
datasets_dir = Path("./datasets")
datasets_dir.mkdir(exist_ok=True) # Create the datasets directory if it doesn't exist
SMI_FILE = datasets_dir / "compounds_10k.smi"

# Download the file
print(f"Downloading {SMI_FILE_URL} to {SMI_FILE}...")
response = requests.get(SMI_FILE_URL)
response.raise_for_status() # Raise an exception for HTTP errors

# Save the file content
with open(SMI_FILE, "wb") as f:
    f.write(response.content)

print(f"Successfully downloaded {SMI_FILE.name} to {SMI_FILE}")

In [ ]:
# Create a perception object first
perception = QueryPharmacophorePerception()
print("Auto-perceivable feature types for a query pharmacophore:")
perception.print_features()

In [ ]:
# Create a molecule (phenol) that is used to build a pharmacophore from
ref_mol = Chem.AddHs(Chem.MolFromSmiles("c1ccccc1O"))
AllChem.EmbedMolecule(ref_mol, randomSeed=0xF00D)
AllChem.UFFOptimizeMolecule(ref_mol)

In [ ]:
ref_mol

In [ ]:
view = py3Dmol.view(
    data=Chem.MolToMolBlock(ref_mol),  # Convert the RDKit molecule for py3Dmol
    style={"stick": {}, "sphere": {"scale": 0.3}}
)
view.zoomTo()

In [ ]:
# Create a first pharmacophore (pharmacophore_1)
pharmacophore_1 = query_pharmacophore_from_molecule(ref_mol, perception, name="phenol")
print(f"\nQuery {pharmacophore_1.get_name()!r} ({len(pharmacophore_1)} features):")
for p in pharmacophore_1:
    print(f"  {p.type.value:<10} center={p.center}")

In [ ]:
mol_from_ph1 = pharmacophore_1.to_mol()

view = py3Dmol.view()
view.addModel(Chem.MolToMolBlock(ref_mol), 'sdf')
view.setStyle({"model": 0}, {'stick': {}, 'sphere': {'radius': 0.3}})
view.addModel(Chem.MolToMolBlock(mol_from_ph1), 'sdf')
view.setStyle({"model": 1},{'stick': {}, 'sphere': {'radius': 0.6}})
view.show()

In [ ]:
# Self-screen sanity check
searcher_1 = PharmacophoreSearch(pharmacophore_1)
print("\nSelf-screen pharmacophore 1:")
print_match_results(searcher_1.screen(ref_mol, progress=False))

In [ ]:
# Build 3D conformers for a SMILES dataset
MAX_COMPOUNDS = 500

smiles_lines = [
    ln.strip()
    for ln in SMI_FILE.read_text(encoding="utf-8").splitlines()
    if ln.strip()
]
if MAX_COMPOUNDS is not None: smiles_lines = smiles_lines[:MAX_COMPOUNDS]

print(f"\nBuilding 3D structures for {len(smiles_lines)} SMILES from {SMI_FILE.name}")
prepared: list[tuple[int, str, Chem.Mol]] = []
parse_fail = embed_fail = 0
with tqdm(smiles_lines, desc="3D structures", unit="mol") as pbar:
    for line_idx, smi in enumerate(pbar):
        mol_3d = Chem.MolFromSmiles(smi)
        if mol_3d is None:
            parse_fail += 1
            continue
        mol_3d = Chem.AddHs(mol_3d)
        if AllChem.EmbedMolecule(mol_3d, randomSeed=0xF00D + line_idx) != 0:
            embed_fail += 1
            continue
        AllChem.UFFOptimizeMolecule(mol_3d)
        prepared.append((line_idx, smi, mol_3d))
        pbar.set_postfix(ready=len(prepared), parse=parse_fail, embed=embed_fail)

print(
    f"Done: {len(prepared)} ligands ready, "
    f"{parse_fail} invalid SMILES, {embed_fail} embed failures"
)

In [ ]:
# Run the screen and report the top hits
# Pharmacophore 1
print("Pharmacophore 1")
hits = searcher_1.screen(prepared, progress=True)
sorted_hits = sort_match_results(hits, sort="descending", key="tanimoto")
print_match_results(sorted_hits, limit=10)

SDF_FILE = Path("./top_hits_ph_1.sdf")
write_hits_sdf(sorted_hits[:10], SDF_FILE, pharmacophore=pharmacophore_1)

PDB_FILE = Path("./top_hits_ph_1.pdb")
write_hits_pdb(sorted_hits[:10], PDB_FILE, pharmacophore=pharmacophore_1)

In [ ]:
# Now visualize the top 5 hits from the top_hits_ph_1.sdf file
suppl = Chem.SDMolSupplier(Path("./top_hits_ph_1.sdf"))

view = py3Dmol.view()
view.addModel(Chem.MolToMolBlock(mol_from_ph1), 'sdf')

counter = 0
for mol in suppl:
    if counter >= 5: break
    else: counter += 1
    if mol is None: continue  # Skip unreadable molecules

    # Process your molecule
    view.addModel(Chem.MolToMolBlock(mol), 'sdf')

view.setStyle({'stick': {}, 'sphere': {'radius': 0.3}})
view.show()


In [ ]:
# Pharmacophore 2: same as pharmacophore_1 but with AROM relaxed to AROM_OR_LIPO.
pharmacophore_2 = pharmacophore_1.copy()
for i, p in enumerate(pharmacophore_2):
    if p.type == PointType.AROM:
        pharmacophore_2.update_point(i, type=PointType.AROM_OR_LIPO)
        break
pharmacophore_2.set_name("phenol-arom-or-lipo")
print(f"\nQuery {pharmacophore_2.get_name()!r} ({len(pharmacophore_2)} features):")
for p in pharmacophore_2:
    print(f"  {p.type.value:<10} center={p.center}")

In [ ]:
# Self screen sanity check
searcher_2 = PharmacophoreSearch(pharmacophore_2)
print("\nSelf-screen pharmacophore 2:")
print_match_results(searcher_2.screen(ref_mol, progress=False))

In [ ]:
# Pharmacophore 2 search
print("Pharmacophore 2")
hits = searcher_2.screen(prepared, progress=True)
sorted_hits = sort_match_results(hits, sort="descending", key="tanimoto")
print_match_results(sorted_hits, limit=10)

SDF_FILE = Path("./top_hits_ph_2.sdf")
write_hits_sdf(sorted_hits[:10], SDF_FILE, pharmacophore=pharmacophore_2)

PDB_FILE = Path("./top_hits_ph_2.pdb")
write_hits_pdb(sorted_hits[:10], PDB_FILE, pharmacophore=pharmacophore_2)